# QueenAgent — Colab

Bu defterin işi kurmak ve sunmak: Drive'ı bağlar → repoyu klonlar → **Flask** arayüzünü servis
eder → bir link basar. Uygulamanın kendisi linkin arkasında çalışıyor.

Projelerin, sohbetlerin ve üretilen dosyaların hepsi **kendi Drive'ında** duruyor — hangi klasörde
olduğunu aşağıdaki CONFIG hücresi söyler (`DRIVE_FOLDER`) ve çalışınca yazdırır. Runtime kapansa da
kaybolmuyorlar.

In [ ]:
# === CONFIG ===
# Drive is mounted on the very first line: the permission window has to appear in the first
# second (NOTEBOOK-STANDARD). One that shows up forty seconds in waits on a user who has already
# walked away.
import os

from google.colab import drive, userdata

drive.mount("/content/drive")

# Everything that gets changed by hand lives here and nowhere else.
DRIVE_FOLDER = "queenAgent"          # proje kökü (MyDrive altında) — adı buradan değiştir
REPO         = "AltanBaysal/Internal-tools"
# Released work, which is what someone else's notebook should be running. A feature branch lives
# only as long as its madde does; one named here goes on working until the day it is deleted, and
# then fails with a sentence that says nothing about when it went stale.
#
# Pointed at feat/v7 for the trial that closes the v7 run -- maddes 150 to 160. THIS MUST GO BACK TO
# main BEFORE THE BRANCH IS MERGED -- test_the_notebook_ships_pointing_at_no_feature_branch is red
# for as long as it says feat/, and that redness is the reminder rather than a fault to be worked
# around.
BRANCH       = "feat/v7"
CLONE_DIR    = "/content/Internal-tools"
APP_DIR      = f"{CLONE_DIR}/queen-agent"
APP_PORT     = 8100                  # backend/config.py ile aynı

# The work lives here. Checked rather than assumed: writing under /content/drive without a mount
# lands on Colab's own disk instead, and that folder dies with the runtime -- the user believes it
# worked and loses everything later.
DRIVE_ROOT = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(DRIVE_ROOT, exist_ok=True)   # ilk koşu oluşturur, sonrakiler aynısını kullanır
assert os.path.isdir(DRIVE_ROOT), (
    f"❌ Proje kökü oluşmadı: {DRIVE_ROOT} — Drive bağlanmamış olabilir, hücreyi tekrar çalıştır"
)

# All three from Colab's Secrets store (🔑 in the left sidebar), never from this cell: the notebook
# is in git, so anything pasted here would sit inside the repository it opens. userdata raises when
# a secret is missing or the notebook was not granted access, and that bare error says nothing -- so
# it is caught and the asserts below say what to do instead.
try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = ""

try:
    XAI_API_KEY = userdata.get("XAI_API_KEY")
except Exception:
    XAI_API_KEY = ""

try:
    DEEPSEEK_API_KEY = userdata.get("DEEPSEEK_API_KEY")
except Exception:
    DEEPSEEK_API_KEY = ""

assert GITHUB_TOKEN, (
    "❌ GITHUB_TOKEN yok — Colab solundaki 🔑 Secrets panelinden 'GITHUB_TOKEN' adıyla ekle "
    "ve bu deftere erişimi aç (fine-grained, yalnız bu depo, Contents: read)."
)

# Stopped here rather than left to the first message. There is no screen to type a key into later,
# so an app started without one can do nothing at all -- and finding that out after everything else
# has come up is finding it out in the worst place.
#
# Both keys, not one of them (Madde 146): the composer offers three models across two providers, so
# a run opened on a single key would promise two of them it cannot answer with.
assert XAI_API_KEY, (
    "❌ XAI_API_KEY yok — Colab solundaki 🔑 Secrets panelinden 'XAI_API_KEY' adıyla kendi xAI "
    "anahtarını ekle ve bu deftere erişimi aç (console.x.ai üzerinden alınır)."
)

assert DEEPSEEK_API_KEY, (
    "❌ DEEPSEEK_API_KEY yok — Colab solundaki 🔑 Secrets panelinden 'DEEPSEEK_API_KEY' adıyla "
    "kendi DeepSeek anahtarını ekle ve bu deftere erişimi aç "
    "(platform.deepseek.com üzerinden alınır)."
)

print(f"✓ Drive bağlı — proje kökü: {DRIVE_ROOT}")
print(f"✓ Dal: {BRANCH}  |  Depo: {REPO}")
print("✓ GITHUB_TOKEN, XAI_API_KEY ve DEEPSEEK_API_KEY Secrets'tan okundu")

In [ ]:
# === Clone ===
# Delete-and-reclone: the local tree is disposable and every run starts from the same place. A pull
# can stop on a merge conflict, and a user looking at that has no way to know what it means.
import shutil
import subprocess

# This cell's paths come from CONFIG. Without the gate, a CONFIG that failed stays invisible until
# something much later breaks for a reason nobody can trace back.
assert "CLONE_DIR" in globals(), "❌ Önce CONFIG hücresini çalıştır"


def _mask(text):
    """Replace the token with <token>, so no output ever carries it."""
    return text.replace(GITHUB_TOKEN, "<token>") if GITHUB_TOKEN else text


if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)

# An argument list, so no shell is involved: a URL that passes through one lands in its history and
# in its log lines. This string carries the token and is never printed.
clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"
result = subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, CLONE_DIR],
    capture_output=True, text=True,
)
if result.returncode != 0:
    # git's own words, masked. A 403 has a dozen causes -- an expired token, a scope that is too
    # narrow, a secret this notebook was never granted, a branch that is gone -- and the notebook
    # knows none of them, so it does not guess.
    raise RuntimeError("❌ Klon başarısız:\n" + _mask(result.stderr.strip() or result.stdout.strip()))

# The app's only third-party dependency. Colab already ships it, so this returns at once -- it is
# written down anyway, because a notebook leaning silently on what Colab happens to include breaks
# with an unreadable ModuleNotFoundError the day that changes.
subprocess.run(["pip", "install", "-q", "flask"], check=True)

# The repo side of this is a test (test_dist_is_committed.py): was the bundle committed. This is the
# other side: did it arrive. Without it a forgotten rebuild reaches the user as a blank page, which
# explains nothing.
DIST = f"{APP_DIR}/frontend/dist/index.html"
assert os.path.exists(DIST), (
    f"❌ Derlenmiş arayüz yok: {DIST} — frontend derlenip commit'lenmemiş (queen-agent/README.md)"
)

print(f"✓ Klon tamam: {CLONE_DIR}  |  dal: {BRANCH}")
print("✓ Flask hazır, derlenmiş arayüz yerinde")

In [ ]:
# === Serve ===
# Starts the app, waits for it to answer, opens an address to it, and prints that address together
# with the one sentence the user needs to hear about it.
import re
import time
import urllib.request

assert "APP_DIR" in globals(), "❌ Önce CONFIG hücresini çalıştır"

APP_LOG = "/content/queenagent.log"
TUNNEL_LOG = "/content/cloudflared.log"

# Re-run safety. Without this a second server is born while the first still holds the port, and
# which one answers becomes anybody's guess -- that exact confusion cost an hour once.
subprocess.run(["pkill", "-f", "main.py"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(2)

# Three things the app learns only from here. QUEENAGENT_ROOT is where it writes; left unset it
# falls back to a home directory, which here is Colab's own disk -- everything would die with the
# runtime and the user would only find out the next day. The two keys are the whole of what it knows
# about them: it saves none, asks for none, and reads these at startup (backend/config.py). Which of
# the two a turn spends is decided by the model picked in the composer, and config.py's table is
# what turns that choice into a key.
# The rest of the environment is carried over rather than replaced: building one from scratch would
# drop PATH along with it.
# cwd is the app folder because main.py does `from backend import config`, and that package only
# resolves from there.
log = open(APP_LOG, "w")
subprocess.Popen(
    ["python", "main.py"], cwd=APP_DIR, stdout=log, stderr=subprocess.STDOUT,
    env={
        **os.environ,
        "QUEENAGENT_ROOT": DRIVE_ROOT,
        "XAI_API_KEY": XAI_API_KEY,
        "DEEPSEEK_API_KEY": DEEPSEEK_API_KEY,
    },
)

for attempt in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{APP_PORT}/api/health", timeout=2)
        print(f"✓ Sunucu ayakta ({(attempt + 1) * 2} sn)")
        break
    except Exception:
        pass
else:
    # The server's own last words. A Flask process dies for a dozen reasons -- a missing package, a
    # held port, a broken import -- and this notebook knows none of them, so it guesses none.
    print("".join(open(APP_LOG).readlines()[-30:]))
    raise RuntimeError("❌ Sunucu 90 sn içinde cevap vermedi — yukarıdaki log'a bak")

# cloudflared rather than Colab's own proxy: that one forwards only GET, and this app creates,
# sends and deletes.
if not os.path.isfile("/content/cloudflared"):
    subprocess.run(
        ["wget", "-q", "-O", "/content/cloudflared",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
        check=True,
    )
    subprocess.run(["chmod", "+x", "/content/cloudflared"], check=True)

subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", f"http://127.0.0.1:{APP_PORT}"],
    stdout=open(TUNNEL_LOG, "w"), stderr=subprocess.STDOUT,
)

link = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists(TUNNEL_LOG):
        found = re.search(r"https://[-\w.]+trycloudflare\.com", open(TUNNEL_LOG).read())
        if found:
            link = found.group(0)
            break
if not link:
    print(open(TUNNEL_LOG).read()[-1000:] if os.path.exists(TUNNEL_LOG) else "(cloudflared log yok)")
    raise RuntimeError("❌ cloudflared linki 30 sn içinde alınamadı — yukarıdaki log'a bak")

# The warning sits beside the link on purpose. There is no login in this app, so whoever holds the
# address holds everything behind it; written in a README instead, that sentence would never reach
# the person who is copying the link right now.
print(f"\n🔗 QueenAgent: {link}\n")
print("⚠️  Bu linkte parola yok — eline geçen herkes projelerini okuyabilir, dosyalarını silebilir")
print("    ve senin API anahtarlarınla istek atabilir. Yalnız güvendiğin birine ver.\n")
print("📡 BU HÜCREYİ KAPATMA — kapanırsa link ölür. Canlı log:\n")

# A finished cell tells Colab there is nothing left to do here, and the runtime gets shut down with
# the tunnel inside it. This keeps the cell open and shows the server's log while it does.
try:
    subprocess.run(["tail", "-n", "+1", "-f", APP_LOG])
except KeyboardInterrupt:
    print("Hücre durduruldu — sunucu hâlâ arka planda. Yeni link için hücreyi tekrar çalıştır.")